# Week 3: Advanced SQL Analysis using Subqueries, CTEs & Window Functions
**Celebal Excellence Internship (CEI) 2026**

Objective: Analyze the Superstore dataset using advanced SQL concepts.

## Step 1: Import Libraries

In [1]:
import pandas as pd
import sqlite3

## Step 2: Load Dataset

In [2]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Step 3: Create SQLite Database

In [3]:
conn = sqlite3.connect("superstore.db")
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)
print("Table created successfully.")

Table created successfully.


## Step 4: Create Required Tables

### Customers Table

In [4]:
query = '''
CREATE TABLE customers AS
SELECT DISTINCT
[Customer ID],[Customer Name],Segment,Country,City,State,Region
FROM superstore_raw;
'''
conn.execute("DROP TABLE IF EXISTS customers")
conn.execute(query)
pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)

,Customer ID,Customer Name,Segment,Country,City,State,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South


### Orders Table

In [5]:
query = '''
CREATE TABLE orders AS
SELECT DISTINCT
[Order ID],[Order Date],[Ship Date],[Ship Mode],[Customer ID]
FROM superstore_raw;
'''
conn.execute("DROP TABLE IF EXISTS orders")
conn.execute(query)
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520
1,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710
4,CA-2017-114412,4/15/2017,4/20/2017,Standard Class,AA-10480


### Products Table

In [6]:
query = '''
CREATE TABLE products AS
SELECT DISTINCT
[Product ID],Category,[Sub-Category],[Product Name]
FROM superstore_raw;
'''
conn.execute("DROP TABLE IF EXISTS products")
conn.execute(query)
pd.read_sql("SELECT * FROM products LIMIT 5;", conn)

,Product ID,Category,Sub-Category,Product Name
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


## Orders Above Average Sales (Subquery)

In [7]:
query = """SELECT * FROM superstore_raw
WHERE Sales > (SELECT AVG(Sales) FROM superstore_raw);"""
pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
3,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152
4,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2355,9974,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,Anne Pryor,Home Office,United States,Los Angeles,...,90032,West,TEC-PH-10004080,Technology,Phones,Avaya 5410 Digital phone,271.9600,5,0.20,27.1960
2356,9977,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,Anne Pryor,Home Office,United States,Los Angeles,...,90032,West,TEC-PH-10002496,Technology,Phones,Cisco SPA301,249.5840,2,0.20,31.1980
2357,9980,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,Anne Pryor,Home Office,United States,Los Angeles,...,90032,West,OFF-BI-10002026,Office Supplies,Binders,Ibico Recycled Linen-Style Covers,437.4720,14,0.20,153.1152
2358,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932


## Highest Sales Order Per Customer (Subquery)

In [8]:
query = """SELECT * FROM superstore_raw s
WHERE Sales=(SELECT MAX(Sales) FROM superstore_raw
WHERE [Customer ID]=s.[Customer ID]);"""
pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
1,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
2,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.1840,9,0.20,85.3092
3,25,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,Emily Burns,Consumer,United States,Orem,...,84057,West,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1044.6300,3,0.00,240.2649
4,28,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,19140,East,FUR-BO-10004834,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.4300,7,0.50,-1665.0522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
790,9926,CA-2015-159534,3/20/2015,3/23/2015,First Class,DH-13075,Dave Hallsten,Corporate,United States,New York City,...,10035,East,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,1087.9360,8,0.20,353.5792
791,9930,CA-2016-129630,9/4/2016,9/4/2016,Same Day,IM-15055,Ionia McGrath,Consumer,United States,San Francisco,...,94122,West,TEC-CO-10003763,Technology,Copiers,Canon PC1060 Personal Laser Copier,2799.9600,5,0.20,944.9865
792,9949,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,Helen Wasserman,Corporate,United States,Indianapolis,...,46203,Central,OFF-AP-10002945,Office Supplies,Appliances,Honeywell Enviracaire Portable HEPA Air Cleane...,2405.2000,8,0.00,793.7160
793,9969,CA-2017-153871,12/11/2017,12/17/2017,Standard Class,RB-19435,Richard Bierner,Consumer,United States,Plainfield,...,7060,East,OFF-BI-10004600,Office Supplies,Binders,Ibico Ibimaster 300 Manual Binding System,735.9800,2,0.00,331.1910


## Total Sales Per Customer (CTE)

In [9]:
query = """WITH customer_sales AS (
SELECT [Customer ID],[Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer ID],[Customer Name])
SELECT * FROM customer_sales;"""
pd.read_sql(query, conn)

,Customer ID,Customer Name,Total_Sales
0,AA-10315,Alex Avila,5563.560
1,AA-10375,Allen Armold,1056.390
2,AA-10480,Andrew Allen,1790.512
3,AA-10645,Anna Andreadi,5086.935
4,AB-10015,Aaron Bergman,886.156
...,...,...,...
788,XP-21865,Xylona Preis,2374.658
789,YC-21895,Yoseph Carroll,5454.350
790,YS-21880,Yana Sorensen,6720.444
791,ZC-21910,Zuschuss Carroll,8025.707


## Customers Above Average Sales (CTE + Subquery)

In [10]:
query = """WITH customer_sales AS (
SELECT [Customer ID],[Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer ID],[Customer Name])
SELECT * FROM customer_sales
WHERE Total_Sales>(SELECT AVG(Total_Sales) FROM customer_sales);"""
pd.read_sql(query, conn)

,Customer ID,Customer Name,Total_Sales
0,AA-10315,Alex Avila,5563.560
1,AA-10645,Anna Andreadi,5086.935
2,AB-10060,Adam Bellavance,7755.620
3,AB-10105,Adrian Barton,14473.571
4,AC-10450,Amy Cox,5527.846
...,...,...,...
289,VW-21775,Victoria Wilson,6134.038
290,WB-21850,William Brown,6160.102
291,YC-21895,Yoseph Carroll,5454.350
292,YS-21880,Yana Sorensen,6720.444


## Rank Customers (Window Function)

In [11]:
query = """SELECT [Customer Name],
SUM(Sales) AS Total_Sales,
RANK() OVER(ORDER BY SUM(Sales) DESC) AS Sales_Rank
FROM superstore_raw
GROUP BY [Customer Name];"""
pd.read_sql(query, conn)

,Customer Name,Total_Sales,Sales_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


## Row Number Within Each Customer

In [12]:
query = """SELECT [Customer Name],[Order ID],Sales,
ROW_NUMBER() OVER(
PARTITION BY [Customer Name]
ORDER BY Sales DESC) AS Order_Number
FROM superstore_raw;"""
pd.read_sql(query, conn)

,Customer Name,Order ID,Sales,Order_Number
0,Aaron Bergman,CA-2016-140935,341.960,1
1,Aaron Bergman,CA-2014-156587,242.940,2
2,Aaron Bergman,CA-2016-140935,221.980,3
3,Aaron Bergman,CA-2014-156587,48.712,4
4,Aaron Bergman,CA-2014-156587,17.940,5
...,...,...,...,...
9989,Zuschuss Donatelli,CA-2017-141481,61.440,5
9990,Zuschuss Donatelli,CA-2014-143336,22.720,6
9991,Zuschuss Donatelli,US-2016-147991,16.720,7
9992,Zuschuss Donatelli,CA-2016-152471,15.984,8


## Top 3 Customers

In [13]:
query = """SELECT * FROM(
SELECT [Customer Name],
SUM(Sales) Total_Sales,
RANK() OVER(ORDER BY SUM(Sales) DESC) Sales_Rank
FROM superstore_raw
GROUP BY [Customer Name])
WHERE Sales_Rank<=3;"""
pd.read_sql(query, conn)

,Customer Name,Total_Sales,Sales_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3


## Final Combined Query

In [14]:
query = """WITH customer_sales AS(
SELECT [Customer ID],[Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer ID],[Customer Name])
SELECT [Customer Name],Total_Sales,
RANK() OVER(ORDER BY Total_Sales DESC) AS Customer_Rank
FROM customer_sales;"""
pd.read_sql(query, conn)

,Customer Name,Total_Sales,Customer_Rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


## Top 5 Customers

In [15]:
query = """SELECT [Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer Name]
ORDER BY Total_Sales DESC
LIMIT 5;"""
pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571


## Bottom 5 Customers

In [16]:
query = """SELECT [Customer Name],SUM(Sales) Total_Sales
FROM superstore_raw
GROUP BY [Customer Name]
ORDER BY Total_Sales ASC
LIMIT 5;"""
pd.read_sql(query, conn)

,Customer Name,Total_Sales
0,Thais Sissman,4.833
1,Lela Donovan,5.304
2,Carl Jackson,16.520
3,Mitch Gastineau,16.739
4,Roy Skaria,22.328


## Customers With One Order

In [17]:
query = """SELECT [Customer Name],COUNT(DISTINCT [Order ID]) Orders
FROM superstore_raw
GROUP BY [Customer Name]
HAVING Orders=1;"""
pd.read_sql(query, conn)

,Customer Name,Orders
0,Anemone Ratner,1
1,Anthony O'Donnell,1
2,Carl Jackson,1
3,Jenna Caffey,1
4,Jocasta Rupert,1
5,Lela Donovan,1
6,Mitch Gastineau,1
7,Patricia Hirasaki,1
8,Ricardo Emerson,1
9,Roland Murray,1


## Highest Order Value Per Customer

In [18]:
query = """SELECT [Customer Name],MAX(Sales) Highest_Order
FROM superstore_raw
GROUP BY [Customer Name];"""
pd.read_sql(query, conn)

,Customer Name,Highest_Order
0,Aaron Bergman,341.960
1,Aaron Hawkins,668.160
2,Aaron Smayling,1439.982
3,Adam Bellavance,4355.168
4,Adam Hart,841.568
...,...,...
788,Xylona Preis,337.088
789,Yana Sorensen,2793.528
790,Yoseph Carroll,2934.330
791,Zuschuss Carroll,1516.200


# Conclusion

## Key Insights
* Performed data analysis on the Superstore dataset using advanced SQL techniques.
* Used Subqueries to identify high-value customers and compare sales performance.
* Applied Common Table Expressions (CTEs) to simplify complex queries and calculate aggregated sales data.
* Implemented Window Functions such as ROW_NUMBER() and RANK() to rank customers and products based on sales.
* Combined CTEs and Window Functions to generate customer-wise sales insights and improve query readability.
* Analyzed business trends including top customers, product performance, and sales distribution.
* Improved SQL query efficiency by organizing complex logic into structured and reusable queries.
* Gained practical experience in solving real-world business problems using SQL analytics.
